In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Define paths
DATA_DIR = "../datasets/"
MODEL_DIR = "../../backend/models/"

In [2]:
print("Loading PhiUSIIL Phishing URL dataset...")
file_path = os.path.join(DATA_DIR, "PhiUSIIL_Phishing_URL_Dataset.csv")

try:
    df = pd.read_csv(file_path)
    print(f"✅ Data loaded successfully. Total rows: {len(df)}")
except FileNotFoundError:
    print(f"❌ Error: Could not find {file_path}. Please check the folder.")

# CRITICAL STEP: Select ONLY features the backend can extract from a raw string instantly.
# We are intentionally dropping HTML/Content features to keep the API extremely fast.
usable_features = [
    'URLLength', 
    'DomainLength', 
    'IsDomainIP', 
    'NoOfSubDomain', 
    'NoOfLettersInURL', 
    'NoOfDegitsInURL', 
    'NoOfEqualsInURL', 
    'NoOfQMarkInURL', 
    'IsHTTPS',
    'label' # The target variable
]

# Filter the dataframe
df_filtered = df[usable_features].dropna()
print(f"✅ Filtered dataset down to {len(usable_features)-1} fast-compute features.")

Loading PhiUSIIL Phishing URL dataset...
✅ Data loaded successfully. Total rows: 235795
✅ Filtered dataset down to 9 fast-compute features.


In [3]:
# Separate features (X) and target (y)
X = df_filtered.drop('label', axis=1)
y = df_filtered['label']

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)} URLs")
print(f"Testing set size: {len(X_test)} URLs")

Training set size: 188636 URLs
Testing set size: 47159 URLs


In [4]:
print("🧠 Training Random Forest Classifier...")
# n_estimators=100 provides a good balance of accuracy and speed
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

rf_model.fit(X_train, y_train)
print("✅ Model training complete.")

# Evaluate accuracy
predictions = rf_model.predict(X_test)
acc = accuracy_score(y_test, predictions)
print(f"\n📊 Model Accuracy on Test Data: {acc * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, predictions))

🧠 Training Random Forest Classifier...
✅ Model training complete.

📊 Model Accuracy on Test Data: 99.77%

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20124
           1       1.00      1.00      1.00     27035

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159



In [5]:
print("💾 Exporting URL model for Mahi's Backend...")

os.makedirs(MODEL_DIR, exist_ok=True)
url_model_path = os.path.join(MODEL_DIR, "url_model.pkl")

# Save the trained model
joblib.dump(rf_model, url_model_path)

print(f"✅ SUCCESS! URL Model saved to: {os.path.abspath(url_model_path)}")

💾 Exporting URL model for Mahi's Backend...
✅ SUCCESS! URL Model saved to: d:\Projects\IDTHP\CyberSentinel-ai\backend\models\url_model.pkl
